# Análisis Cuantitativo de Vulnerabilidades y Dependencias
Este notebook contiene el análisis de los SBOMs, dependencias y vulnerabilidades descubiertas en los repositorios seleccionados de la organización **`encode`**.

## Reproducibilidad
Para reproducir este análisis:
1. Haber ejecutado los scripts de obtención de datos desde el entorno Dev Container (`ciberseguridad_2026`).
2. Los resultados en formato JSON deben residir en las carpetas respectivas dentro de `ciberseguridad_2026/data/results/`.


In [7]:
import pandas as pd
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual
sns.set_theme(style="whitegrid")


In [8]:
RESULTS_DIR = Path("ciberseguridad_2026/data/results")
if not RESULTS_DIR.exists():
    RESULTS_DIR = Path("data/results") # Si se corre desde dentro de ciberseguridad_2026
if not RESULTS_DIR.exists():
    RESULTS_DIR = Path("entregable_final/ciberseguridad_2026/data/results") # Si se corre desde afuera
    
print(f"Buscando resultados en: {RESULTS_DIR.absolute()}")
print(f"¿Existe el directorio?: {RESULTS_DIR.exists()}")


Buscando resultados en: e:\UFRO\5to-2026\mineria-repositorio\Analisis - sboms\entregable_final\ciberseguridad_2026\data\results
¿Existe el directorio?: True


## 1. Análisis de Vulnerabilidades en Dependencias (Grype)

In [9]:
grype_data = []

if RESULTS_DIR.exists():
    # Iterar solo sobre los reportes finales (ignorando los raw)
    for f in RESULTS_DIR.glob("*-grype.json"):
        with open(f, "r", encoding="utf-8") as file:
            data = json.load(file)
            repo_name = f.stem.replace("-grype", "")
            
            for match in data.get("matches", []):
                vuln = match.get("vulnerability", {})
                artifact = match.get("artifact", {})
                
                grype_data.append({
                    "repo": repo_name,
                    "vulnerability_id": vuln.get("id"),
                    "severity": vuln.get("severity"),
                    "package_name": artifact.get("name"),
                    "package_version": artifact.get("version"),
                    "type": artifact.get("type")
                })

df_grype = pd.DataFrame(grype_data)

if not df_grype.empty:
    display(df_grype.head())
else:
    print("No se encontraron vulnerabilidades en los JSONs cargados.")


No se encontraron vulnerabilidades en los JSONs cargados.


In [10]:
if not df_grype.empty:
    plt.figure(figsize=(10, 6))
    ax = sns.countplot(data=df_grype, x="severity", order=["Low", "Medium", "High", "Critical"], palette="Reds")
    plt.title("Distribución de Severidad de Vulnerabilidades en Dependencias")
    plt.ylabel("Cantidad de Vulnerabilidades")
    plt.xlabel("Severidad")
    plt.show()
    
    # Top 5 paquetes con más vulnerabilidades
    top_vuln_packages = df_grype['package_name'].value_counts().head(5)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=top_vuln_packages.values, y=top_vuln_packages.index, palette="viridis")
    plt.title("Top 5 Dependencias con más Vulnerabilidades")
    plt.xlabel("Total de CVEs")
    plt.ylabel("Paquete")
    plt.show()


## 2. Análisis de Código Fuente Estático (CodeQL)

In [11]:
codeql_data = []

if RESULTS_DIR.exists():
    for f in RESULTS_DIR.glob("*-codeql.json"):
        with open(f, "r", encoding="utf-8") as file:
            data = json.load(file)
            repo_name = f.stem.replace("-codeql", "")
            
            for run in data.get("runs", []):
                for result in run.get("results", []):
                    codeql_data.append({
                        "repo": repo_name,
                        "rule_id": result.get("ruleId"),
                        "message": result.get("message", {}).get("text"),
                        "severity": result.get("level", "warning")
                    })

df_codeql = pd.DataFrame(codeql_data)

if not df_codeql.empty:
    display(df_codeql.head())
else:
    print("No se encontraron alertas en los JSONs de CodeQL.")



No se encontraron alertas en los JSONs de CodeQL.


In [12]:
if not df_codeql.empty:
    plt.figure(figsize=(10, 6))
    sns.countplot(data=df_codeql, y="rule_id", order=df_codeql['rule_id'].value_counts().index, palette="magma")
    plt.title("Reglas (CWEs) violadas más frecuentemente en Código Fuente")
    plt.xlabel("Cantidad de Alertas")
    plt.ylabel("Regla (CWE/CodeQL)")
    plt.show()


## 3. Conclusiones
*(Elaborar las conclusiones una vez generados los gráficos)*
- **Estado de las Dependencias**: ¿Cuántas dependencias críticas tenemos?
- **Salud del Código (CodeQL)**: ¿Qué reglas de seguridad saltan más frecuentemente y por qué?
